# Final Held-Out Test Evaluation

Comparison of three ART classification architectures:

1. MLP with frozen fastText embeddings
2. Standard two-layer GRU
3. PairwiseGRU with shared hypothesis scoring

All models use their previously selected seed-42 checkpoints.

The held-out test set is used only for final evaluation.
No model training, hyperparameter tuning, or checkpoint
selection will be performed using test results.

## Setup and Checkpoint Selection

We prepare the final evaluation of the MLP, standard GRU, and PairwiseGRU using their previously selected seed-42 checkpoints.

All models use the same frozen 300-dimensional fastText embeddings.

The checkpoint selection was completed using validation data before evaluating the held-out test set.

No training, hyperparameter tuning, or model selection is performed using test results.

In [1]:
# ============================================================
# Final Evaluation — Setup and Frozen Checkpoint Selection
# ============================================================

import sys
import inspect
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from datasets import load_dataset

# Locate repository root
repo_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "project1" / "artifacts" / "fasttext_art.pt").exists()
    ),
    None
)

assert repo_root is not None, "Repository root not found"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from project1.src.models import (
    EmbeddingMLP,
    EmbeddingGRU,
    PairwiseGRU
)
from project1.src.data_loader import create_dataloader
from project1.src.training import evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load frozen fastText artifact
artifact_dir = repo_root / "project1" / "artifacts"

saved = torch.load(
    artifact_dir / "fasttext_art.pt",
    map_location="cpu",
    weights_only=True
)

vocab = saved["vocab"]
embedding_weights = saved["embedding_weights"]

assert embedding_weights.shape == (15038, 300)

# Checkpoint selection fixed BEFORE test evaluation
checkpoints = {
    "MLP": (
        "mlp_multiseed_s42_drop0p2_lr0p0003_"
        "t6fumkja_best.pt"
    ),
    "Standard GRU": (
        "gru_multiseed_h128_lr0p001_drop0p0_"
        "seed42_cuda_hvxtunb8_best.pt"
    ),
    "PairwiseGRU": (
        "pairwise_gru_h128_l2_lr0p001_drop0p0_"
        "seed42_cuda_z6d879wl_best.pt"
    )
}

# Previously established validation accuracies
validation_accuracy = {
    "MLP": 0.650,
    "Standard GRU": 0.685,
    "PairwiseGRU": 0.711
}

print("=== FINAL EVALUATION SETUP ===")
print("Device:", device)
print("Vocabulary size:", len(vocab))
print("Embedding shape:", tuple(embedding_weights.shape))

print("\n=== SELECTED CHECKPOINTS ===")

for model_name, filename in checkpoints.items():
    path = artifact_dir / filename

    assert path.is_file(), f"Missing checkpoint: {path}"

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    assert isinstance(state, dict)

    print(f"\n{model_name}")
    print("File:", filename)
    print("Validation accuracy:",
          f"{validation_accuracy[model_name]:.2%}")
    print("State dictionary entries:", len(state))
    print("First parameter names:", list(state)[:5])
    print("Checkpoint: PASS")

print("\n=== MODEL CONSTRUCTORS ===")

for model_cls in (EmbeddingMLP, EmbeddingGRU, PairwiseGRU):
    print(f"{model_cls.__name__}:")
    print(" ", inspect.signature(model_cls))

print("\nSetup: PASS")
print("Checkpoint selection: LOCKED")
print("Held-out test set: NOT ACCESSED")


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== FINAL EVALUATION SETUP ===
Device: cuda
Vocabulary size: 15038
Embedding shape: (15038, 300)

=== SELECTED CHECKPOINTS ===

MLP
File: mlp_multiseed_s42_drop0p2_lr0p0003_t6fumkja_best.pt
Validation accuracy: 65.00%
State dictionary entries: 5
First parameter names: ['embedding.weight', 'classifier.0.weight', 'classifier.0.bias', 'classifier.2.weight', 'classifier.2.bias']
Checkpoint: PASS

Standard GRU
File: gru_multiseed_h128_lr0p001_drop0p0_seed42_cuda_hvxtunb8_best.pt
Validation accuracy: 68.50%
State dictionary entries: 13
First parameter names: ['embedding.weight', 'gru.weight_ih_l0', 'gru.weight_hh_l0', 'gru.bias_ih_l0', 'gru.bias_hh_l0']
Checkpoint: PASS

PairwiseGRU
File: pairwise_gru_h128_l2_lr0p001_drop0p0_seed42_cuda_z6d879wl_best.pt
Validation accuracy: 71.10%
State dictionary entries: 13
First parameter names: ['embedding.weight', 'gru.weight_ih_l0', 'gru.weight_hh_l0', 'gru.bias_ih_l0', 'gru.bias_hh_l0']
Checkpoint: PASS

=== MODEL CONSTRUCTORS ===
EmbeddingMLP:
  (emb

## Restore and Verify All Models

We restore the three previously selected checkpoints and verify their performance on the validation set before final test evaluation.

For each model, we:
- Reconstruct the original architecture and hyperparameters.
- Load the saved checkpoint with strict parameter matching.
- Verify that the fastText embeddings remain frozen.
- Reproduce the previously recorded validation accuracy.

The held-out test set is not accessed during this verification.

In [2]:
# ============================================================
# Final Evaluation — Restore and Verify All Models
# ============================================================

# Load validation data ONLY (test remains untouched)
val_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="validation"
)

val_loader = create_dataloader(
    val_data,
    vocab,
    batch_size=32,
    shuffle=False
)


# ------------------------------------------------------------
# Frozen fastText embedding factory
# ------------------------------------------------------------

def create_embedding_layer():
    return nn.Embedding.from_pretrained(
        embedding_weights.detach().cpu().clone(),
        freeze=True,
        padding_idx=vocab["[PAD]"]
    )


# ------------------------------------------------------------
# Model factories — original selected configurations
# ------------------------------------------------------------

def create_mlp():
    return EmbeddingMLP(
        embedding_layer=create_embedding_layer(),
        pad_id=vocab["[PAD]"],
        sep_id=vocab["[SEP]"],
        hidden_dim=128,
        dropout=0.2
    ).to(device)


def create_standard_gru():
    return EmbeddingGRU(
        embedding_layer=create_embedding_layer(),
        pad_id=vocab["[PAD]"],
        hidden_dim=128,
        num_layers=2,
        dropout=0.0
    ).to(device)


def create_pairwise_gru():
    return PairwiseGRU(
        embedding_layer=create_embedding_layer(),
        pad_id=vocab["[PAD]"],
        sep_id=vocab["[SEP]"],
        hidden_dim=128,
        num_layers=2,
        dropout=0.0
    ).to(device)


model_factories = {
    "MLP": create_mlp,
    "Standard GRU": create_standard_gru,
    "PairwiseGRU": create_pairwise_gru
}

criterion = nn.CrossEntropyLoss()

restored_models = {}
validation_results = {}


# ------------------------------------------------------------
# Restore checkpoints and reproduce validation accuracies
# ------------------------------------------------------------

for model_name, factory in model_factories.items():

    print(f"\n{'=' * 55}")
    print(f"MODEL: {model_name}")
    print(f"{'=' * 55}")

    model = factory()

    checkpoint_path = (
        artifact_dir / checkpoints[model_name]
    )

    state_dict = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    # Strict loading: fail if parameters do not match
    model.load_state_dict(
        state_dict,
        strict=True
    )

    model.eval()

    # Verify frozen fastText
    assert not model.embedding.weight.requires_grad

    # Evaluate only on validation
    metrics = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    expected_accuracy = validation_accuracy[model_name]
    actual_accuracy = metrics["accuracy"]

    print(f"Expected accuracy: {expected_accuracy:.2%}")
    print(f"Restored accuracy: {actual_accuracy:.2%}")
    print(f"Validation loss: {metrics['loss']:.4f}")

    assert abs(
        actual_accuracy - expected_accuracy
    ) < 1e-6, (
        f"{model_name}: Validation accuracy mismatch!"
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print("Trainable parameters:", trainable_params)
    print("Frozen embeddings: PASS")
    print("Checkpoint restoration: PASS")

    restored_models[model_name] = model

    validation_results[model_name] = {
        "accuracy": actual_accuracy,
        "loss": metrics["loss"],
        "parameters": trainable_params
    }


# ============================================================
# Summary
# ============================================================

print("\n=== FINAL CHECKPOINT VERIFICATION ===")

for name, result in validation_results.items():
    print(
        f"{name:15s} | "
        f"Accuracy: {result['accuracy']:.2%} | "
        f"Loss: {result['loss']:.4f}"
    )

assert len(restored_models) == 3

print("\nAll three checkpoints restored: PASS")
print("All validation accuracies reproduced: PASS")
print("Held-out test set: NOT ACCESSED")



MODEL: MLP
Expected accuracy: 65.00%
Restored accuracy: 65.00%
Validation loss: 0.6796
Trainable parameters: 153986
Frozen embeddings: PASS
Checkpoint restoration: PASS

MODEL: Standard GRU
Expected accuracy: 68.50%
Restored accuracy: 68.50%
Validation loss: 0.5945
Trainable parameters: 280962
Frozen embeddings: PASS
Checkpoint restoration: PASS

MODEL: PairwiseGRU
Expected accuracy: 71.10%
Restored accuracy: 71.10%
Validation loss: 0.5941
Trainable parameters: 280833
Frozen embeddings: PASS
Checkpoint restoration: PASS

=== FINAL CHECKPOINT VERIFICATION ===
MLP             | Accuracy: 65.00% | Loss: 0.6796
Standard GRU    | Accuracy: 68.50% | Loss: 0.5945
PairwiseGRU     | Accuracy: 71.10% | Loss: 0.5941

All three checkpoints restored: PASS
All validation accuracies reproduced: PASS
Held-out test set: NOT ACCESSED


### Checkpoint Verification Results

| Model | Validation accuracy | Validation loss | Trainable parameters |
|---|---:|---:|---:|
| MLP | 65.00% | 0.6796 | 153,986 |
| Standard GRU | 68.50% | 0.5945 | 280,962 |
| PairwiseGRU | 71.10% | 0.5941 | 280,833 |

All checkpoints loaded successfully, and all validation accuracies were reproduced exactly.

The PairwiseGRU achieved the highest validation accuracy. The selected checkpoints are now ready for held-out test evaluation without further training or model selection.

## Final Held-Out Test Evaluation

We evaluate the three previously selected models on the official held-out test set (1,532 examples).

No additional training, hyperparameter tuning, or checkpoint selection is performed.

We compare:
- Accuracy and balanced accuracy
- Macro-F1 and cross-entropy loss
- Confusion matrices and per-class recall

This provides a final comparison of generalization performance.

In [3]:
# ============================================================
# FINAL HELD-OUT TEST EVALUATION
# ============================================================

import torch.nn.functional as F

# ------------------------------------------------------------
# 1. Load held-out test data
# ------------------------------------------------------------

test_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="test"
)

assert "label" in test_data.column_names, (
    "Test labels are not available."
)

# Original ART labels: 1 = H1, 2 = H2
assert set(test_data["label"]) == {1, 2}

test_loader = create_dataloader(
    test_data,
    vocab,
    batch_size=32,
    shuffle=False
)

print("=== HELD-OUT TEST DATA ===")
print("Test samples:", len(test_data))
print("Original labels:", set(test_data["label"]))
print("Test DataLoader: PASS")


# ------------------------------------------------------------
# 2. Evaluation function
# ------------------------------------------------------------

def evaluate_final_model(model, loader):

    model.eval()

    all_predictions = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.inference_mode():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"]
            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                lengths=lengths
            )

            predictions = logits.argmax(dim=1)

            total_loss += F.cross_entropy(
                logits,
                labels,
                reduction="sum"
            ).item()

            total_samples += labels.size(0)

            all_predictions.append(predictions.cpu())
            all_labels.append(labels.cpu())

    y_pred = torch.cat(all_predictions)
    y_true = torch.cat(all_labels)

    # Verify DataLoader label mapping and example order
    expected_labels = (
        torch.tensor(test_data["label"], dtype=torch.long) - 1
    )

    assert torch.equal(y_true, expected_labels)

    # Confusion matrix:
    # Rows = true label, columns = predicted label
    confusion = torch.bincount(
        y_true * 2 + y_pred,
        minlength=4
    ).reshape(2, 2)

    assert confusion.sum().item() == len(test_data)

    # Accuracy
    accuracy = (
        (y_pred == y_true).float().mean().item()
    )

    # Per-class recall
    recall = (
        confusion.diag().float()
        / confusion.sum(dim=1).clamp(min=1)
    )

    # Per-class precision
    precision = (
        confusion.diag().float()
        / confusion.sum(dim=0).clamp(min=1)
    )

    # Per-class F1
    f1 = (
        2 * precision * recall
        / (precision + recall).clamp(min=1e-12)
    )

    return {
        "accuracy": accuracy,
        "balanced_accuracy": recall.mean().item(),
        "macro_f1": f1.mean().item(),
        "loss": total_loss / total_samples,
        "h1_recall": recall[0].item(),
        "h2_recall": recall[1].item(),
        "confusion": confusion
    }


# ------------------------------------------------------------
# 3. Evaluate all three frozen models
# ------------------------------------------------------------

test_results = {}

for model_name, model in restored_models.items():

    metrics = evaluate_final_model(
        model,
        test_loader
    )

    test_results[model_name] = metrics

    print(f"\n{'=' * 55}")
    print(f"FINAL TEST RESULT — {model_name}")
    print(f"{'=' * 55}")

    print(
        f"Validation accuracy: "
        f"{validation_accuracy[model_name]:.2%}"
    )

    print(
        f"Test accuracy: "
        f"{metrics['accuracy']:.2%}"
    )

    print(
        f"Balanced accuracy: "
        f"{metrics['balanced_accuracy']:.2%}"
    )

    print(
        f"Macro-F1: "
        f"{metrics['macro_f1']:.4f}"
    )

    print(
        f"Test loss: "
        f"{metrics['loss']:.4f}"
    )

    print("\nConfusion matrix:")
    print("             Pred H1  Pred H2")
    print(
        f"True H1      "
        f"{metrics['confusion'][0, 0]:7d}  "
        f"{metrics['confusion'][0, 1]:7d}"
    )
    print(
        f"True H2      "
        f"{metrics['confusion'][1, 0]:7d}  "
        f"{metrics['confusion'][1, 1]:7d}"
    )

    print("\nPer-class recall:")
    print(f"H1: {metrics['h1_recall']:.2%}")
    print(f"H2: {metrics['h2_recall']:.2%}")


# ------------------------------------------------------------
# 4. Final comparison
# ------------------------------------------------------------

print("\n=== FINAL HELD-OUT TEST COMPARISON ===")

print(
    f"{'Model':15s} "
    f"{'Validation':>12s} "
    f"{'Test':>10s} "
    f"{'Balanced':>10s} "
    f"{'Macro-F1':>10s}"
)

for name, metrics in test_results.items():

    print(
        f"{name:15s} "
        f"{validation_accuracy[name]:12.2%} "
        f"{metrics['accuracy']:10.2%} "
        f"{metrics['balanced_accuracy']:10.2%} "
        f"{metrics['macro_f1']:10.4f}"
    )

print("\nAll models evaluated: PASS")
print("No training performed: PASS")
print("Final held-out evaluation: COMPLETE")


=== HELD-OUT TEST DATA ===
Test samples: 1532
Original labels: {1, 2}
Test DataLoader: PASS

FINAL TEST RESULT — MLP
Validation accuracy: 65.00%
Test accuracy: 51.57%
Balanced accuracy: 51.54%
Macro-F1: 0.5154
Test loss: 1.0770

Confusion matrix:
             Pred H1  Pred H2
True H1          412      369
True H2          373      378

Per-class recall:
H1: 52.75%
H2: 50.33%

FINAL TEST RESULT — Standard GRU
Validation accuracy: 68.50%
Test accuracy: 54.05%
Balanced accuracy: 54.09%
Macro-F1: 0.5404
Test loss: 0.9039

Confusion matrix:
             Pred H1  Pred H2
True H1          405      376
True H2          328      423

Per-class recall:
H1: 51.86%
H2: 56.32%

FINAL TEST RESULT — PairwiseGRU
Validation accuracy: 71.10%
Test accuracy: 53.98%
Balanced accuracy: 53.98%
Macro-F1: 0.5398
Test loss: 1.2057

Confusion matrix:
             Pred H1  Pred H2
True H1          421      360
True H2          345      406

Per-class recall:
H1: 53.91%
H2: 54.06%

=== FINAL HELD-OUT TEST COMPARIS

### Final Test Results

| Model | Validation accuracy | Test accuracy | Macro-F1 |
|---|---:|---:|---:|
| MLP | 65.00% | 51.57% | 0.5154 |
| Standard GRU | 68.50% | 54.05% | 0.5404 |
| PairwiseGRU | 71.10% | 53.98% | 0.5398 |

### Interpretation

The Standard GRU achieved the highest test accuracy (54.05%), narrowly ahead of the PairwiseGRU (53.98%).

Although the PairwiseGRU achieved the highest validation accuracy, this advantage did not transfer to the test set.

All three models experienced substantial validation-to-test accuracy decreases:

- MLP: 13.43 percentage points
- Standard GRU: 14.45 percentage points
- PairwiseGRU: 17.12 percentage points

**Conclusion:** The results suggest a generalization problem affecting all three architectures. The PairwiseGRU's higher validation accuracy did not translate into better held-out test performance.

The test accuracies are only slightly above the 50% chance baseline, highlighting the difficulty of generalizing to unseen examples.

## Train / Validation / Test Split Audit

We investigate potential dataset differences that could explain the large validation-to-test performance drop.

We compare:
- Dataset size and class balance
- Average number of content tokens
- Unknown-token rates
- Exact observation-pair overlap between splits

This is a read-only diagnostic. No training or model selection is performed.

In [4]:

# ============================================================
# Diagnostic — Train / Validation / Test Split Audit
# READ-ONLY: No training or model selection
# ============================================================

from collections import Counter

train_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="train"
)

splits = {
    "train": train_data,
    "validation": val_data,
    "test": test_data
}

pad_id = vocab["[PAD]"]
sep_id = vocab["[SEP]"]
unk_id = vocab["[UNK]"]

def split_statistics(dataset):
    loader = create_dataloader(
        dataset,
        vocab,
        batch_size=64,
        shuffle=False
    )

    total_tokens = 0
    unknown_tokens = 0

    for batch in loader:
        ids = batch["input_ids"]

        content_mask = (
            (ids != pad_id) &
            (ids != sep_id)
        )

        total_tokens += content_mask.sum().item()

        unknown_tokens += (
            (ids == unk_id) & content_mask
        ).sum().item()

    labels = Counter(dataset["label"])

    return {
        "samples": len(dataset),
        "H1_labels": labels[1],
        "H2_labels": labels[2],
        "H1_percent": labels[1] / len(dataset) * 100,
        "avg_tokens": total_tokens / len(dataset),
        "unk_percent": (
            100 * unknown_tokens / total_tokens
            if total_tokens else 0
        )
    }


print("=== DATASET SPLIT COMPARISON ===")

for name, dataset in splits.items():
    stats = split_statistics(dataset)

    print(f"\n{name.upper()}")
    print("Samples:", stats["samples"])
    print(f"H1 labels: {stats['H1_labels']} "
          f"({stats['H1_percent']:.2f}%)")
    print("H2 labels:", stats["H2_labels"])
    print(f"Average content tokens: "
          f"{stats['avg_tokens']:.2f}")
    print(f"Unknown-token rate: "
          f"{stats['unk_percent']:.2f}%")


# Check exact overlap of observation pairs
def observation_pairs(dataset):
    return {
        (
            row["observation_1"].strip().lower(),
            row["observation_2"].strip().lower()
        )
        for row in dataset
    }


train_pairs = observation_pairs(train_data)
val_pairs = observation_pairs(val_data)
test_pairs = observation_pairs(test_data)

print("\n=== OBSERVATION PAIR OVERLAP ===")
print("Train / Validation:",
      len(train_pairs & val_pairs))
print("Train / Test:",
      len(train_pairs & test_pairs))
print("Validation / Test:",
      len(val_pairs & test_pairs))

print("\nDataset audit: COMPLETE")


=== DATASET SPLIT COMPARISON ===

TRAIN
Samples: 16801
H1 labels: 8461 (50.36%)
H2 labels: 8340
Average content tokens: 39.09
Unknown-token rate: 0.38%

VALIDATION
Samples: 1000
H1 labels: 508 (50.80%)
H2 labels: 492
Average content tokens: 39.56
Unknown-token rate: 1.63%

TEST
Samples: 1532
H1 labels: 781 (50.98%)
H2 labels: 751
Average content tokens: 39.53
Unknown-token rate: 1.63%

=== OBSERVATION PAIR OVERLAP ===
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

Dataset audit: COMPLETE


### Dataset Audit Results

| Metric | Train | Validation | Test |
|---|---:|---:|---:|
| Examples | 16,801 | 1,000 | 1,532 |
| H1 labels | 50.36% | 50.80% | 50.98% |
| Average content tokens | 39.09 | 39.56 | 39.53 |
| Unknown-token rate | 0.38% | 1.63% | 1.63% |

**Observation-pair overlap:**
- Train / Validation: 0
- Train / Test: 0
- Validation / Test: 0

### Interpretation

Class balance and average content length are very similar across all three splits.

Validation and test also have identical unknown-token rates (1.63%), suggesting that unknown-token frequency alone does not explain their performance difference.

No exact observation-pair overlap was detected between splits. However, this check does not rule out other forms of data overlap.

**Conclusion:** These basic dataset statistics do not explain the substantial validation-to-test accuracy decrease. Further investigation of hypothesis characteristics is needed.

## Hypothesis Similarity Across Splits

We investigate whether the two candidate hypotheses differ systematically between training, validation, and test data.

We compare:
- Jaccard similarity between hypothesis token sets
- Absolute hypothesis length differences
- Number of identical hypothesis pairs

This analysis helps identify potential distribution shifts that could contribute to the validation-to-test performance drop.

In [5]:
# ============================================================
# Diagnostic — Hypothesis Similarity Across Splits
# READ-ONLY: No training or checkpoint changes
# ============================================================

from project1.src.preprocessing import tokenize
import numpy as np

def hypothesis_statistics(dataset):
    similarities = []
    length_differences = []
    identical_pairs = 0

    for example in dataset:
        h1 = tokenize(example["hypothesis_1"])
        h2 = tokenize(example["hypothesis_2"])

        s1, s2 = set(h1), set(h2)

        union = s1 | s2
        similarity = (
            len(s1 & s2) / len(union)
            if union else 1.0
        )

        similarities.append(similarity)
        length_differences.append(abs(len(h1) - len(h2)))

        if h1 == h2:
            identical_pairs += 1

    return {
        "mean_similarity": np.mean(similarities),
        "median_similarity": np.median(similarities),
        "mean_length_difference": np.mean(length_differences),
        "identical_pairs": identical_pairs
    }


print("=== HYPOTHESIS DIFFICULTY COMPARISON ===")

for name, dataset in splits.items():
    stats = hypothesis_statistics(dataset)

    print(f"\n{name.upper()}")
    print(f"Mean Jaccard similarity: "
          f"{stats['mean_similarity']:.3f}")
    print(f"Median similarity: "
          f"{stats['median_similarity']:.3f}")
    print(f"Mean hypothesis length difference: "
          f"{stats['mean_length_difference']:.2f} tokens")
    print(f"Identical hypotheses: "
          f"{stats['identical_pairs']}")

print("\nDiagnostic: COMPLETE")


=== HYPOTHESIS DIFFICULTY COMPARISON ===

TRAIN
Mean Jaccard similarity: 0.602
Median similarity: 0.667
Mean hypothesis length difference: 1.19 tokens
Identical hypotheses: 10

VALIDATION
Mean Jaccard similarity: 0.617
Median similarity: 0.667
Mean hypothesis length difference: 1.08 tokens
Identical hypotheses: 1

TEST
Mean Jaccard similarity: 0.225
Median similarity: 0.167
Mean hypothesis length difference: 3.10 tokens
Identical hypotheses: 0

Diagnostic: COMPLETE


### Hypothesis Similarity Results

| Metric | Train | Validation | Test |
|---|---:|---:|---:|
| Mean Jaccard similarity | 0.602 | 0.617 | 0.225 |
| Median similarity | 0.667 | 0.667 | 0.167 |
| Mean length difference | 1.19 | 1.08 | 3.10 |
| Identical hypothesis pairs | 10 | 1 | 0 |

### Interpretation

The training and validation splits have similar hypothesis characteristics. However, the test set differs substantially:

- **Lower lexical similarity:** Test hypotheses share considerably fewer tokens than validation hypotheses.
- **Larger length differences:** Test hypotheses differ in length by 3.10 tokens on average, compared with 1.08 in validation.
- **Distribution shift:** The models were selected on validation examples whose hypothesis characteristics differ from those of the test examples.

This provides a plausible explanation for the substantial validation-to-test accuracy decrease observed across all three models.

### Conclusion

The test set exhibits a clear shift in hypothesis similarity and length differences.

These differences may contribute to the models' poor generalization, although this diagnostic alone does not establish causality.

The results highlight the importance of evaluating models beyond a single validation distribution.